# Challenge 5 · Find the Hidden State
**Tier 2 - Quantum Algorithms and Applications** · **Estimated time: 40-60 minutes** (an estimate, not a guarantee) · *Participant notebook*

> A vault accepts exactly one n-bit code. Checking codes one by one takes ~2^n tries; Grover's algorithm needs only ~sqrt(2^n) oracle calls. Build the search and watch the amplitude of the hidden code grow.

### Learning objectives
- Implement a phase oracle that marks one basis state.
- Implement the diffuser (inversion about the mean).
- Explain phase marking and amplitude amplification geometrically.
- Choose the number of iterations and observe what happens when you overshoot.

### Prerequisites
- Challenges 1-3 or equivalent
- Multi-controlled gates
- Superposition and phase

### What you will write
| Function | Status |
|---|---|
| `phase_oracle` | required |
| `diffuser` | required |
| `optimal_iterations` | required |
| `grover_circuit` | required |
| `phase_oracle_multi` | bonus (optional) |

### Acceptance criteria
Your solution is accepted when all **core** and **edge-case** requirements hold:
- `phase_oracle(marked)` flips the sign of exactly the marked basis state (up to global phase), in Qiskit bit order.
- `diffuser(n)` implements the inversion about the uniform superposition, 2|s><s| - I (up to global phase).
- `optimal_iterations(n, num_marked)` returns an iteration count that maximises the ideal success probability.
- `grover_circuit` reaches the theoretical success probability sin^2((2k+1)theta) for every marked state and iteration count.
- Invalid marked strings, sizes and iteration counts raise ValueError; returned circuits have the documented width, no measurements, and are independent objects.

**Bonus (optional, never required to pass):**
- `phase_oracle_multi` handles several marked states and Grover with M solutions matches theory.

**How this notebook works.** Run the cells from top to bottom. Cells marked **TODO** are yours. After each part run the **public checks**: they tell you what works and what does not. The instructors run additional hidden tests, so also try your own edge cases. Hints go from subtle to explicit - open them one at a time.

In [ ]:
# Setup - safe to re-run (works in Google Colab and in a local Jupyter session)
import importlib, importlib.util, pathlib, subprocess, sys

if importlib.util.find_spec('qiskit_fest') is None:
    here = pathlib.Path.cwd().resolve()
    # the project is the working folder, one of its parents, or (Colab: working folder /content) a folder right below it
    candidates = [here, *here.parents, *sorted(p for p in here.iterdir() if p.is_dir())]
    root = next((p for p in candidates
                 if (p / 'pyproject.toml').is_file() and (p / 'src' / 'qiskit_fest').is_dir()), None)
    if root is None:
        raise RuntimeError("Open this notebook from inside the qiskit-fall-fest-challenges folder "
                           "(see README.md, section 'Colab').")
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(root)])
    except subprocess.CalledProcessError:
        print('pip could not install the project (no internet?) - continuing with the libraries that are already installed.')
    # a running kernel does not re-read the .pth file that pip has just written, so make the package importable here and now
    sys.path.insert(0, str(root / 'src'))
    importlib.invalidate_caches()

from qiskit_fest.checks import run_public_checks, if_ready
from qiskit_fest.env import print_environment
print_environment()

## The problem
A vault accepts exactly one n-bit code (the **marked** bit-string). Trying codes one by one needs about N/2 tries on average, N = 2ⁿ.
**Grover's algorithm** finds it with about √N oracle calls. You will build it for a handful of qubits and watch the amplitude of the hidden code grow:

1. `phase_oracle(marked)` - **phase marking**: flip the sign of the amplitude of the marked state, leave all others alone.
2. `diffuser(n)` - **amplitude amplification**: reflect all amplitudes about their mean, so the sign flip becomes a *growth* of the marked amplitude.
3. `optimal_iterations(n, num_marked)` - how many (oracle, diffuser) rounds to run.
4. `grover_circuit(marked, iterations)` - the complete search circuit.
5. *(Bonus)* `phase_oracle_multi(marked_states)` - several marked states at once.

### Function contracts
**Bit order:** `marked` is a string of `'0'`/`'1'` in **Qiskit order** - the *rightmost* character is qubit 0 - so the marked basis state has index `int(marked, 2)`. The number of qubits is `n = len(marked)`.

| Function | Input | Output | Must raise `ValueError` when |
|---|---|---|---|
| `phase_oracle(marked)` | string of 0/1, length n ≥ 1 | n-qubit circuit (**no ancillas, no measurements**) whose unitary is diagonal with −1 on the marked state and +1 elsewhere (up to a global phase) | `marked` is not a non-empty string of `'0'`/`'1'` |
| `diffuser(n)` | `int` n ≥ 1 | n-qubit circuit implementing 2&#124;s⟩⟨s&#124; − I (up to a global phase), &#124;s⟩ = uniform superposition | n is not an integer ≥ 1 |
| `optimal_iterations(n, num_marked=1)` | `int` n ≥ 1, `int` 1 ≤ num_marked < 2ⁿ | non-negative `int` k that maximises the ideal success probability among k = 0 … ⌈π/(4θ)⌉, where θ = arcsin√(M/N) | invalid n or num_marked (not an integer, < 1, or ≥ 2ⁿ) |
| `grover_circuit(marked, iterations=None)` | as above; `iterations=None` means `optimal_iterations(n, 1)` | n-qubit circuit (**no measurements**): H on all qubits, then `iterations` times (oracle, diffuser) | invalid `marked`; `iterations` not a non-negative integer |
| *(bonus)* `phase_oracle_multi(marked_states)` | non-empty sequence of distinct bit-strings of equal length | n-qubit circuit flipping the sign of every listed state | empty list, duplicates, mixed lengths, invalid strings |

**Tolerances.** Operators and states are compared exactly (up to a global phase) with absolute tolerance 1e-9. "Success probability" always means the *exact* probability of the marked outcome.

## Concept: why it works (a rotation in a plane)
Let $|w\rangle$ be the marked state and $|s\rangle=\frac1{\sqrt N}\sum_x|x\rangle$ the uniform state. Then $|s\rangle=\sin\theta\,|w\rangle+\cos\theta\,|s'\rangle$ with $\sin\theta=\sqrt{M/N}$ and $|s'\rangle$ the uniform state over the *unmarked* items.
* The **oracle** reflects the state about $|s'\rangle$ (it flips the sign of the $|w\rangle$ part).
* The **diffuser** $2|s\rangle\langle s|-I$ reflects about $|s\rangle$.
* Two reflections make a **rotation by 2θ** towards $|w\rangle$. After $k$ iterations the angle is $(2k+1)\theta$, so
$$P_{\rm success}(k)=\sin^2\big((2k+1)\theta\big).$$
It peaks when $(2k+1)\theta\approx\pi/2$, i.e. $k\approx\frac{\pi}{4\theta}-\frac12\approx\frac\pi4\sqrt{N/M}$ - and then **falls again**: overshooting rotates past the target.

**Building blocks.** A multi-controlled-Z flips the sign of $|1\ldots1\rangle$ only. To mark another string, flip (with X) the qubits whose target bit is 0 before and after. The diffuser is "H on all qubits, flip the sign of $|0\ldots0\rangle$, H on all qubits".

In [ ]:
import numpy as np
from qiskit import QuantumCircuit

In [ ]:
from qiskit_fest.challenges.c05_grover import (
    EXAMPLE_MARKED, plot_amplitude_steps, plot_success_curve, success_probability, theoretical_success)
from qiskit_fest.viz import draw_circuit

print("Example marked string (non-palindromic on purpose):", EXAMPLE_MARKED, "-> basis-state index", int(EXAMPLE_MARKED, 2))
print("Ideal success probability for n=3 after k = 0, 1, 2, 3 iterations:",
      [round(theoretical_success(3, 1, k), 4) for k in range(4)])

## Part 1 · The phase oracle  *(TODO)*
Complete `phase_oracle(marked)`. Remember the **bit order**: character *i* from the **right** of `marked` belongs to qubit *i*.

<details><summary>💡 Hint 1 (subtle)</summary>

A multi-controlled Z flips the sign of |1…1⟩ and nothing else. How can you make the marked string "look like" |1…1⟩ for a moment?
</details>

<details><summary>💡 Hint 2</summary>

Use X gates on the qubits where the marked bit is `0`, apply the multi-controlled Z, then undo the X gates. Check the order of characters against the order of qubits - a reversed string is the classic bug, and it hides on palindromes like `"101"`.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Skeleton: `for i, bit in enumerate(reversed(marked)):` → X on qubit `i` where `bit == "0"`; then a multi-controlled Z on all qubits (for example `h` on the last qubit, `mcx(controls, target)`, `h` on the last qubit, or `mcp(np.pi, controls, target)`); then the same X gates again. For `n = 1` the multi-controlled Z is simply `z`.
</details>

In [ ]:
def phase_oracle(marked: str) -> QuantumCircuit:
    """n-qubit circuit (n = len(marked), no ancillas) that flips the sign of the basis state `marked` (Qiskit bit order)."""
    # TODO: validate `marked` (ValueError), then build the oracle.
    raise NotImplementedError("TODO: implement phase_oracle()")

In [ ]:
with if_ready():
    draw_circuit(phase_oracle(EXAMPLE_MARKED), f"Phase oracle marking |{EXAMPLE_MARKED}>")

In [ ]:
report = run_public_checks(5, globals(), keyword="phase_oracle")

## Part 2 · The diffuser  *(TODO)*
Complete `diffuser(n)`: the unitary $2|s\rangle\langle s|-I$ on n qubits (a global phase of −1 is fine).

<details><summary>💡 Hint 1 (subtle)</summary>

Reflecting about the uniform state |s⟩ = H^n|0…0⟩ is the same as: change basis with H, reflect about |0…0⟩, change back.
</details>

<details><summary>💡 Hint 2</summary>

"Reflect about |0…0⟩" means "flip the sign of every state except |0…0⟩" - equivalently, up to a global −1, "flip the sign of |0…0⟩ only". You just built an oracle that flips the sign of one chosen state.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Skeleton: H on all qubits → X on all qubits → multi-controlled Z → X on all qubits → H on all qubits (for `n = 1` the multi-controlled Z is simply `z`).
</details>

In [ ]:
def diffuser(n: int) -> QuantumCircuit:
    """n-qubit circuit implementing 2|s><s| - I (up to global phase), |s> = uniform superposition. ValueError unless n is an int >= 1."""
    # TODO: validate n (ValueError), then build H - (flip |0...0>) - H.
    raise NotImplementedError("TODO: implement diffuser()")

In [ ]:
with if_ready():
    plot_amplitude_steps(EXAMPLE_MARKED, phase_oracle, diffuser)

In [ ]:
report = run_public_checks(5, globals(), keyword="diffuser")

## Part 3 · How many iterations?  *(TODO)*
Complete `optimal_iterations(n, num_marked=1)`: with θ = arcsin√(M/N) return the integer k that **maximises** $\sin^2((2k+1)\theta)$ among k = 0 … ⌈π/(4θ)⌉.
Do not just use √N: for N = 8 the best k is 2, for N = 16 it is 3.

<details><summary>💡 Hint 1 (subtle)</summary>

You want $(2k+1)\theta$ as close to $\pi/2$ as possible. Solve for k, then think about rounding.
</details>

<details><summary>💡 Hint 2</summary>

Alternatively do not solve anything: evaluate $\sin^2((2k+1)\theta)$ for the first few k and keep the best.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

`theta = np.arcsin(np.sqrt(___ / ___))`, then either `k = int(___(np.pi / (4 * theta)))` or `max(range(___), key=lambda k: np.sin((2 * k + 1) * theta) ** 2)`. Validate n and num_marked first.
</details>

In [ ]:
def optimal_iterations(n: int, num_marked: int = 1) -> int:
    """Best number of Grover iterations for N = 2**n items and `num_marked` marked items (ValueError on invalid input)."""
    # TODO: validate the arguments, compute theta = arcsin(sqrt(M/N)) and choose k.
    raise NotImplementedError("TODO: implement optimal_iterations()")

In [ ]:
with if_ready():
    for n in (2, 3, 4, 5, 6):
        print(f"n={n} (N={2**n:3d}): optimal iterations = {optimal_iterations(n)}   (rounding sqrt(N) would suggest {round(2**(n / 2))})")

In [ ]:
report = run_public_checks(5, globals(), keyword="optimal_iterations")

## Part 4 · The Grover circuit  *(TODO)*
Complete `grover_circuit(marked, iterations=None)`: H on all qubits, then `iterations` times your oracle followed by your diffuser. `iterations=None` means "use `optimal_iterations(n, 1)`". No measurements.
After that, the plot shows the success probability of *your* circuit for every number of iterations next to the theory.

<details><summary>💡 Hint 1 (subtle)</summary>

Order matters: the uniform superposition is prepared **once**; each iteration is "oracle, then diffuser".
</details>

<details><summary>💡 Hint 2</summary>

`QuantumCircuit.compose(other, inplace=True)` appends another circuit of the same width. `iterations=0` must give just the uniform superposition.
</details>

<details><summary>💡 Hint 3 (explicit)</summary>

Skeleton: validate `iterations` (None → optimal; otherwise a non-negative int, else `ValueError`); `qc.___(range(n))`; `for _ in range(___):` compose the oracle, then the diffuser. Build the oracle and diffuser circuits once, outside the loop.
</details>

In [ ]:
def grover_circuit(marked: str, iterations: int | None = None) -> QuantumCircuit:
    """H on all qubits, then `iterations` x (phase_oracle, diffuser). iterations=None -> optimal_iterations(n, 1). No measurements."""
    # TODO: validate, prepare the uniform superposition, apply the iterations.
    raise NotImplementedError("TODO: implement grover_circuit()")

In [ ]:
with if_ready():
    plot_success_curve(grover_circuit, EXAMPLE_MARKED)
    qc = grover_circuit(EXAMPLE_MARKED)
    print(f"default iterations -> P(success) = {success_probability(qc, EXAMPLE_MARKED):.4f}")

In [ ]:
report = run_public_checks(5, globals(), keyword="grover_circuit")

## Extension · What happens with other iteration counts?  *(optional, not graded)*
Grover's algorithm is **not** "more iterations, better result": every iteration rotates the state by 2θ in the |w⟩–|s'⟩ plane, so the success probability sin²((2k+1)θ) rises, peaks and falls again. Explore it with your own circuit:

1. **Find the peak by simulation.** For n = 3, 4, 5 loop over k = 0, 1, 2, … with `grover_circuit(marked, k)` and `success_probability`. Where is the first maximum? Compare it with `optimal_iterations(n)`.
2. **Overshoot.** Run 2·k\* iterations. What is the success probability now, and which picture in the rotation plane explains it?
3. **A very small search space.** For n = 2 (N = 4) one iteration finds the marked string with probability exactly 1 - and two iterations only 1/4. Check it and explain why N = 4 is special (θ = π/6).
4. **The wrong number of marked states.** With M = 2 marked strings but k chosen for M = 1, what do you get? (Use `phase_oracle_multi` if you solved the bonus, or the formula sin²((2k+1)θ_M) with θ_M = arcsin√(M/N).)

The cell below prints the success probability of **your** circuit for k = 0 … 2k\*+1, so that you can start from real numbers.

In [ ]:
with if_ready():
    print(f"{'n':>2} {'k*':>3} {'P(k*)':>7} {'P(2k*)':>7}   success probability of YOUR circuit for k = 0 ... 2k*+1")
    for n in (3, 4, 5):
        marked = "1" * n
        k_star = optimal_iterations(n)
        curve = [success_probability(grover_circuit(marked, k), marked) for k in range(2 * k_star + 2)]
        print(f"{n:>2} {k_star:>3} {curve[k_star]:>7.3f} {curve[2 * k_star]:>7.3f}   " + " ".join(f"{p:.2f}" for p in curve))
    print("N = 4 (n = 2):", "   ".join(f"k={k}: {success_probability(grover_circuit('11', k), '11'):.3f}" for k in range(4)))

## Bonus · Several marked states  *(optional)*
Complete `phase_oracle_multi(marked_states)`: flip the sign of **every** listed state (they must be distinct, of equal length). With M marked states of N = 2ⁿ, the success probability (summed over the marked states) after k iterations is $\sin^2((2k+1)\theta_M)$ with $\theta_M=\arcsin\sqrt{M/N}$.
Then run Grover with `optimal_iterations(n, M)` (your Part 2 and 3 functions already handle M).

<details><summary>💡 Hint</summary>

Phase flips on different basis states commute: apply your single-state oracle once per marked state and compose them.
</details>

In [ ]:
def phase_oracle_multi(marked_states) -> QuantumCircuit:
    """n-qubit circuit flipping the sign of every bit-string in `marked_states` (distinct, equal length; ValueError otherwise)."""
    # TODO (bonus): validate and compose one single-state oracle per marked state.
    raise NotImplementedError("TODO (bonus): implement phase_oracle_multi()")

In [ ]:
with if_ready():
    marked = ["011", "101"]
    qc = QuantumCircuit(3)
    qc.h(range(3))
    k = optimal_iterations(3, len(marked))
    for _ in range(k):
        qc.compose(phase_oracle_multi(marked), inplace=True)
        qc.compose(diffuser(3), inplace=True)
    total = sum(success_probability(qc, m) for m in marked)
    print(f"marked {marked}, k = {k}: total success probability = {total:.4f}  (theory {theoretical_success(3, len(marked), k):.4f})")

In [ ]:
report = run_public_checks(5, globals(), keyword="bonus")

## Reflection
1. Run the success-probability plot with more iterations than optimal. Why does the probability go *down* again? Draw the picture of the rotation in the |w⟩–|s'⟩ plane.
2. Grover needs ~√N oracle calls and classical search ~N/2. Why does this "only" give a *quadratic* speed-up, and what does the oracle cost in a real problem?
3. Suppose you do not know how many marked states exist. How would that affect your choice of k?
4. Your oracle is built from a multi-controlled gate. How does its gate count grow with n?

### Hand-in
When you are done, save your functions as a file (or simply hand in this notebook).

In [ ]:
# Optional: save your functions as a stand-alone file you can hand in
from qiskit_fest.export import export_submission
export_submission(globals(), 5)